# NTPDase1 Molecular Dynamics Simulation Pipeline

Companion notebook to: *Conserved Hinge-Region Mutations Produce Geometry-Specific Structural Effects in Leishmania NTPDase1: A Cross-Species Computational Target Validation Study*

**Author:** Daniel Szolc 

## Overview

This notebook runs all-atom molecular dynamics simulations of wild-type and hinge-mutant NTPDase1 from *Leishmania infantum* and *Leishmania major*. Four independent 25 ns simulations were performed on Google Colab Pro+ using an NVIDIA A100 GPU.

## Systems

- L. infantum wild type
- L. infantum hinge mutant (P40A, S43A, P44A, Y49F)
- L. major wild type
- L. major hinge mutant (same four substitutions)

## Simulation Protocol

| Parameter | Value |
|-----------|-------|
| Force field | AMBER14 (amber14-all.xml) |
| Water model | TIP3P (amber14/tip3pfb.xml) |
| Temperature | 310 K (Langevin thermostat) |
| Pressure | 1 atm (Monte Carlo barostat) |
| Timestep | 2 fs |
| Water box padding | 0.8 nm |
| Electrostatics | PME, 1.0 nm cutoff |
| Constraints | HBonds |
| Langevin friction | 1.0 ps⁻¹ |
| Equilibration | 100 ps NPT |
| Production | 25 ns per system |
| Frame save interval | 100 ps |
| Platform | OpenCL (A100 80GB) |

## Disulfide Bond Detection

The function automatically detects disulfide bonds by measuring SG–SG distances. Any cysteine pair within 0.25 nm is renamed CYX and explicitly bonded. Four disulfide bonds are detected in NTPDase1: CYS45–CYS124, CYS267–CYS292, CYS306–CYS311, CYS361–CYS377.

## Requirements

- Google Colab Pro+ (A100 GPU recommended)
- 200GB Google Drive storage
- Input PDB files in `/content/drive/MyDrive/NTPDase1_MD/`

## 1. Environment Setup

In [ ]:
import sys
import os
import types

# Install required packages for the correct Python interpreter
os.system(f"{sys.executable} -m pip install pdbfixer mdtraj -q")

# Colab-specific: block broken XTC module which has NumPy 2.0 incompatibility
fake_xtcfile = types.ModuleType('openmm.app.xtcfile')
fake_xtcfile.XTCFile = None
sys.modules['openmm.app.xtcfile'] = fake_xtcfile
sys.modules['openmm.app.internal.xtc_utils'] = types.ModuleType('openmm.app.internal.xtc_utils')

# Import OpenMM
import openmm as mm
import openmm.app as app
import openmm.unit as unit

# Report available platforms
for i in range(mm.Platform.getNumPlatforms()):
    print(mm.Platform.getPlatform(i).getName())
print(f"OpenMM version: {mm.__version__}")

# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

# Import PDBFixer
from pdbfixer import PDBFixer
print("PDBFixer imported successfully")

## 2. Simulation Parameters and Input Files

In [ ]:
# Input and output folders
pdb_folder = '/content/drive/MyDrive/NTPDase1_MD'
output_folder = '/content/drive/MyDrive/NTPDase1_MD/results'
os.makedirs(output_folder, exist_ok=True)

# Simulation parameters
temperature = 310 * unit.kelvin
pressure = 1.0 * unit.atmospheres
timestep = 2.0 * unit.femtoseconds
equilibration_steps = 50000       # 100 ps
production_steps = 12500000       # 25 ns
report_interval = 50000           # save frame every 100 ps

# Input structures
structures = {
    'Linfantum_WT':  'WildtypeInfactum.pdb',
    'Linfantum_MUT': '4MUTInfactum.pdb',
    'Lmajor_WT':     'WtLmajopr.pdb',
    'Lmajor_MUT':    'Lmajor4mutataions.pdb'
}

print(f"Production run: {production_steps * timestep.value_in_unit(unit.femtoseconds) / 1e6} ns per structure")
print("Setup complete")

## 3. Simulation Function

The `run_simulation` function performs the full workflow for one system:

1. Structure preparation using PDBFixer (missing atoms and hydrogens)
2. Automatic disulfide bond detection by SG–SG distance
3. Renaming disulfide cysteines to CYX and adding explicit bonds
4. Solvation in a rectangular water box (0.8 nm padding)
5. System creation with AMBER14 force field and PME electrostatics
6. Energy minimisation
7. 100 ps NPT equilibration
8. 25 ns production run with DCD, log and checkpoint reporters

In [ ]:
def run_simulation(name, pdb_file):
    """Run a complete MD simulation for one NTPDase1 system.

    Parameters
    ----------
    name : str
        Identifier for output files (e.g. 'Linfantum_WT').
    pdb_file : str
        PDB filename inside pdb_folder.

    Returns
    -------
    openmm.app.Simulation
        The completed simulation object.
    """
    from pdbfixer import PDBFixer
    import itertools

    print(f"\n{'='*50}")
    print(f"Starting simulation: {name}")
    print(f"{'='*50}")

    # 1. Structure preparation
    pdb_path = f"{pdb_folder}/{pdb_file}"
    print("Preparing structure with PDBFixer...")
    fixer = PDBFixer(filename=pdb_path)
    fixer.findMissingResidues()
    fixer.findMissingAtoms()
    fixer.addMissingAtoms()
    fixer.addMissingHydrogens(7.4)

    forcefield = app.ForceField('amber14-all.xml', 'amber14/tip3pfb.xml')
    modeller = app.Modeller(fixer.topology, fixer.positions)

    # 2. Disulfide bond detection
    sg_atoms = {}
    for res in modeller.topology.residues():
        if res.name == 'CYS':
            for atom in res.atoms():
                if atom.name == 'SG':
                    sg_atoms[res] = atom

    positions = modeller.positions
    for r1, r2 in itertools.combinations(sg_atoms.keys(), 2):
        p1 = positions[sg_atoms[r1].index].value_in_unit(unit.nanometer)
        p2 = positions[sg_atoms[r2].index].value_in_unit(unit.nanometer)
        d = sum((a - b) ** 2 for a, b in zip(p1, p2)) ** 0.5
        if d < 0.25:  # 2.5 A threshold
            r1.name = 'CYX'
            r2.name = 'CYX'
            modeller.topology.addBond(sg_atoms[r1], sg_atoms[r2])
            print(f"Disulfide bond: CYS{r1.id}-CYS{r2.id}")

    # 3. Solvation
    modeller.addSolvent(forcefield, padding=0.8 * unit.nanometers)
    print("Structure prepared and solvated")

    # 4. System creation
    system = forcefield.createSystem(
        modeller.topology,
        nonbondedMethod=app.PME,
        nonbondedCutoff=1.0 * unit.nanometers,
        constraints=app.HBonds
    )

    # NPT ensemble
    barostat = mm.MonteCarloBarostat(pressure, temperature)
    system.addForce(barostat)

    # Langevin integrator
    integrator = mm.LangevinMiddleIntegrator(
        temperature,
        1.0 / unit.picoseconds,
        timestep
    )

    # GPU platform
    platform = mm.Platform.getPlatformByName('OpenCL')

    simulation = app.Simulation(
        modeller.topology, system, integrator, platform
    )
    simulation.context.setPositions(modeller.positions)

    # 5. Energy minimisation
    print("Running energy minimisation...")
    simulation.minimizeEnergy()
    print("Minimisation complete")

    # 6. Equilibration
    print("Running equilibration (100 ps)...")
    simulation.context.setVelocitiesToTemperature(temperature)
    simulation.step(equilibration_steps)
    print("Equilibration complete")

    # 7. Set up reporters for production
    dcd_file = f"{output_folder}/{name}_trajectory.dcd"
    log_file = f"{output_folder}/{name}_log.txt"
    chk_file = f"{output_folder}/{name}_checkpoint.chk"

    simulation.reporters.append(
        app.DCDReporter(dcd_file, report_interval)
    )
    simulation.reporters.append(
        app.StateDataReporter(
            log_file, report_interval,
            step=True, time=True,
            potentialEnergy=True,
            temperature=True,
            progress=True,
            totalSteps=production_steps
        )
    )
    simulation.reporters.append(
        app.StateDataReporter(
            sys.stdout, report_interval * 10,
            step=True, time=True,
            progress=True,
            totalSteps=production_steps
        )
    )
    # Checkpoint every 500,000 steps (1 ns) for crash recovery
    simulation.reporters.append(
        app.CheckpointReporter(chk_file, 500000)
    )

    # 8. Production run
    print("Running production simulation (25 ns)...")
    print("Checkpoint saved every 1 ns")
    simulation.step(production_steps)

    # Save final structure
    final_pdb = f"{output_folder}/{name}_final.pdb"
    positions = simulation.context.getState(getPositions=True).getPositions()
    app.PDBFile.writeFile(simulation.topology, positions, open(final_pdb, 'w'))

    print(f"\nSimulation complete: {name}")
    print(f"Trajectory saved: {dcd_file}")

    return simulation

print("Simulation function defined")

## 4. Resume Function (Optional)

This function resumes an interrupted simulation from the last saved checkpoint. Requires the checkpoint file `.chk` and existing log/DCD files from the original run.

In [ ]:
def resume_simulation(name, pdb_file):
    """Resume an interrupted simulation from the last checkpoint."""
    from pdbfixer import PDBFixer
    import itertools

    print(f"\n{'='*50}")
    print(f"Resuming simulation: {name}")
    print(f"{'='*50}")

    pdb_path = f"{pdb_folder}/{pdb_file}"
    log_file = f"{output_folder}/{name}_log.txt"
    dcd_file = f"{output_folder}/{name}_trajectory.dcd"
    chk_file = f"{output_folder}/{name}_checkpoint.chk"

    # Read completed steps from log
    with open(log_file, 'r') as f:
        lines = f.readlines()
        last_line = lines[-1].strip()
        completed_steps = int(last_line.split(',')[1])

    remaining_steps = production_steps - completed_steps
    print(f"Simulation reached step {completed_steps}")
    print(f"Remaining steps: {remaining_steps}")

    # Rebuild system exactly as before
    fixer = PDBFixer(filename=pdb_path)
    fixer.findMissingResidues()
    fixer.findMissingAtoms()
    fixer.addMissingAtoms()
    fixer.addMissingHydrogens(7.4)

    forcefield = app.ForceField('amber14-all.xml', 'amber14/tip3pfb.xml')
    modeller = app.Modeller(fixer.topology, fixer.positions)

    sg_atoms = {}
    for res in modeller.topology.residues():
        if res.name == 'CYS':
            for atom in res.atoms():
                if atom.name == 'SG':
                    sg_atoms[res] = atom

    positions = modeller.positions
    for r1, r2 in itertools.combinations(sg_atoms.keys(), 2):
        p1 = positions[sg_atoms[r1].index].value_in_unit(unit.nanometer)
        p2 = positions[sg_atoms[r2].index].value_in_unit(unit.nanometer)
        d = sum((a - b) ** 2 for a, b in zip(p1, p2)) ** 0.5
        if d < 0.25:
            r1.name = 'CYX'
            r2.name = 'CYX'
            modeller.topology.addBond(sg_atoms[r1], sg_atoms[r2])

    modeller.addSolvent(forcefield, padding=0.8 * unit.nanometers)

    system = forcefield.createSystem(
        modeller.topology,
        nonbondedMethod=app.PME,
        nonbondedCutoff=1.0 * unit.nanometers,
        constraints=app.HBonds
    )
    system.addForce(mm.MonteCarloBarostat(pressure, temperature))

    integrator = mm.LangevinMiddleIntegrator(
        temperature, 1.0 / unit.picoseconds, timestep
    )
    platform = mm.Platform.getPlatformByName('OpenCL')

    simulation = app.Simulation(modeller.topology, system, integrator, platform)

    # Load checkpoint (positions, velocities, box vectors)
    simulation.loadCheckpoint(chk_file)
    print("Checkpoint loaded")

    # Append to existing files
    simulation.reporters.append(
        app.DCDReporter(dcd_file, report_interval, append=True)
    )
    simulation.reporters.append(
        app.StateDataReporter(
            log_file, report_interval,
            step=True, time=True,
            potentialEnergy=True,
            temperature=True,
            progress=True,
            totalSteps=production_steps,
            append=True
        )
    )
    simulation.reporters.append(
        app.CheckpointReporter(chk_file, 500000)
    )

    print(f"Running remaining {remaining_steps} steps...")
    simulation.step(remaining_steps)

    final_pdb = f"{output_folder}/{name}_final.pdb"
    positions = simulation.context.getState(getPositions=True).getPositions()
    app.PDBFile.writeFile(simulation.topology, positions, open(final_pdb, 'w'))

    print(f"Simulation complete: {name}")

    return simulation

print("Resume function defined")

## 5. Run Simulations

Run one simulation at a time (~10-12 hours each on A100). To run all four sequentially, use the loop cell after the individual cells.

In [ ]:
sim1 = run_simulation('Linfantum_WT', structures['Linfantum_WT'])

In [ ]:
sim2 = run_simulation('Linfantum_MUT', structures['Linfantum_MUT'])

In [ ]:
sim3 = run_simulation('Lmajor_WT', structures['Lmajor_WT'])

In [ ]:
sim4 = run_simulation('Lmajor_MUT', structures['Lmajor_MUT'])

## Alternative: Sequential Loop

Runs all four simulations back-to-back. Total runtime approximately 40-48 hours on A100.

In [ ]:
# for name, pdb in structures.items():
#     run_simulation(name, pdb)
# print('\nALL 4 SIMULATIONS COMPLETE')

## Output Files

For each system four files are produced in `/content/drive/MyDrive/NTPDase1_MD/results/`:

- `{name}_trajectory.dcd` — full production trajectory (250 frames per 25 ns)
- `{name}_log.txt` — energy, temperature and progress log
- `{name}_final.pdb` — final structure with water and ions
- `{name}_checkpoint.chk` — last checkpoint for resume

Analysis (RMSD, RMSF, statistical testing) is performed in the companion notebook `NTPDase1_MD_Analysis.ipynb`.